# Exponentiated Gradient — ACSIncome (Illinois, 2018)

Evaluate an in-processing fairness constraint for Logistic Regression and Random Forest. Models are trained on the training split and evaluated on validation data. The held-out test set and bootstrap confidence intervals are reserved for `06_final_evaluation.ipynb`.

## Data and models

Use the same data split and preprocessing pipelines as the preceding experiments. `SEX` is excluded from the predictors but used as the sensitive attribute for the fairness constraint.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from fairlearn.metrics import demographic_parity_difference, equal_opportunity_difference, false_positive_rate
from fairlearn.reductions import ExponentiatedGradient, TruePositiveRateParity
from sklearn import set_config
from sklearn.metrics import accuracy_score, balanced_accuracy_score

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_acs_income
from src.experiment import prepare_data_splits
from src.models import create_logistic_regression, create_random_forest

In [8]:
X, y, A = load_acs_income()
X_train, X_val, X_test, y_train, y_val, y_test, A_train, A_val, A_test = (
    prepare_data_splits(X, y, A)
)

## Fairness-constrained training

`ExponentiatedGradient` fits a mixture of classifiers under a true-positive-rate parity constraint. The sensitive attribute is supplied to the reduction during training; it is not included in the underlying model features. Both classifiers use the same constraint and optimization settings.

In [ ]:
set_config(enable_metadata_routing=False)

base_models = {
    "Logistic Regression": create_logistic_regression(),
    "Random Forest": create_random_forest(),
}

models = {}
for name, estimator in base_models.items():
    model = ExponentiatedGradient(
        estimator=estimator,
        constraints=TruePositiveRateParity(),
        eps=0.02,
        max_iter=50,
        sample_weight_name="classifier__sample_weight",
    )
    model.fit(X_train, y_train, sensitive_features=A_train)
    models[name] = model

## Validation results

Evaluate accuracy, balanced accuracy, and absolute group disparities. EOD is the equal opportunity difference; DPD is the demographic parity difference; FPR Gap measures the difference in false positive rates. Predictions from the learned classifier mixture use a fixed random seed.

In [ ]:
def evaluate_validation(name, model):
    predictions = np.asarray(model.predict(X_val, random_state=42))
    groups = np.asarray(A_val)
    labels = np.asarray(y_val)
    male = groups == 1
    female = groups == 2

    return {
        "Model": name,
        "Strategy": "ExponentiatedGradient",
        "Accuracy": accuracy_score(labels, predictions),
        "Balanced Accuracy": balanced_accuracy_score(labels, predictions),
        "EOD": abs(equal_opportunity_difference(
            labels, predictions, sensitive_features=groups
        )),
        "DPD": abs(demographic_parity_difference(
            labels, predictions, sensitive_features=groups
        )),
        "FPR Gap": abs(
            false_positive_rate(labels[male], predictions[male])
            - false_positive_rate(labels[female], predictions[female])
        ),
    }

In [ ]:
validation_results = pd.DataFrame([
    evaluate_validation(name, model) for name, model in models.items()
])
display(validation_results.round(4))

,Model,Strategy,Accuracy,Balanced Accuracy,EOD,DPD,FPR Gap
0,Logistic Regression,ExponentiatedGradient,0.8008,0.7889,0.0001,0.1247,0.0391
1,Random Forest,ExponentiatedGradient,0.7859,0.7694,0.0288,0.1307,0.0427


## Save results and limitations

The validation table is saved for comparison with the other mitigation strategies. The reduction learns a randomized classifier mixture, so its realized predictions depend on the random seed. Validation metrics are preliminary; final test metrics and bootstrap confidence intervals will be computed together for all strategies in `06_final_evaluation.ipynb`.

In [ ]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
validation_results.to_csv(
    results_dir / "exponentiated_gradient_validation.csv", index=False
)